# Demo: Fixing `TD_PLOT` Size Limits with `ts_compress_to`

This notebook generates four noisy trend series with 1,000,000 points each,
tries to visualize the raw 4M-row table with `TD_PLOT`, then uses the
`td_ts_compressor` package to materialize a visually compressed table for
plotting. The purpose is visual fidelity, not analytical feature extraction.

## 1. Setup

Run `00_install_ts_compress_to_operator.ipynb` first so the table operator is
installed in Teradata. Credentials are loaded from `.env` with `load_dotenv`.

In [ ]:
from __future__ import annotations

import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import teradatasql
from dotenv import load_dotenv
from IPython.display import Image, display
from teradataml import copy_to_sql, create_context, remove_context

import td_ts_compressor
from td_ts_compressor import (
    __version__ as TD_TS_COMPRESSOR_VERSION,
    CompressorSpec,
    TeradataTimeSeriesCompressor,
    visual_fidelity,
)

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "pyproject.toml").exists() else NOTEBOOK_DIR.parent
ENV_PATH = REPO_ROOT / ".env"
ASSET_DIR = REPO_ROOT / "notebooks" / "assets"
ASSET_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_TD_TS_COMPRESSOR_VERSION = "0.1.10"
if TD_TS_COMPRESSOR_VERSION != EXPECTED_TD_TS_COMPRESSOR_VERSION:
    raise RuntimeError(
        "Notebook expects td-ts-compressor "
        f"{EXPECTED_TD_TS_COMPRESSOR_VERSION}, but imported {TD_TS_COMPRESSOR_VERSION}. "
        "Reinstall with: python -m pip install -e . and restart the kernel."
    )

load_dotenv(ENV_PATH)
required = ["TD_HOST", "TD_USERNAME", "TD_PASSWORD"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Missing required environment variables: {', '.join(missing)}")

OPERATOR_DATABASE = (
    os.getenv("TD_OPERATOR_DATABASE")
    or os.getenv("TD_COMPRESSOR_DATABASE")
    or os.getenv("TD_DATABASE")
)

print(f"Repository root: {REPO_ROOT}")
print(f"td_ts_compressor version: {TD_TS_COMPRESSOR_VERSION}")
print(f"td_ts_compressor path: {Path(td_ts_compressor.__file__).resolve()}")
print("Operator database:", OPERATOR_DATABASE or "current session database")


def td_plot_image_bytes(rows):
    for row in rows:
        for value in row:
            if isinstance(value, (bytes, bytearray, memoryview)):
                return bytes(value)
    raise ValueError("TD_PLOT returned no bytes-like image payload")


def save_td_plot_png(rows, path: Path) -> Path:
    path.write_bytes(td_plot_image_bytes(rows))
    return path


## 2. Connect to Teradata

In [ ]:
connect_args = {
    "host": os.environ["TD_HOST"],
    "user": os.environ["TD_USERNAME"],
    "password": os.environ["TD_PASSWORD"],
}
context_args = {
    "host": os.environ["TD_HOST"],
    "username": os.environ["TD_USERNAME"],
    "password": os.environ["TD_PASSWORD"],
}
if os.getenv("TD_LOGMECH"):
    connect_args["logmech"] = os.environ["TD_LOGMECH"]
    context_args["logmech"] = os.environ["TD_LOGMECH"]
if os.getenv("TD_DATABASE"):
    context_args["database"] = os.environ["TD_DATABASE"]

con = teradatasql.connect(**connect_args)
td_context = create_context(**context_args)
cur = con.cursor()
if os.getenv("TD_DATABASE"):
    cur.execute(f"DATABASE {os.environ['TD_DATABASE']}")
cur.execute("SELECT DATABASE")
DATA_DATABASE = cur.fetchone()[0]
print("Current database:", DATA_DATABASE)


## 3. Generate and Load Four 1M-Point Series

The source table is generated in chunks to avoid holding all rows in a single
local dataframe. Uploads are batched through `teradataml.copy_to_sql()`, which
can use FastLoad automatically for larger pandas dataframes. Each series
combines a trend, low-frequency wave, and random noise.

In [ ]:
SOURCE_TABLE = "input_td_plot_extension_4m"
COMPRESSED_TABLE = "plot_td_plot_extension_compressed"
ROWS_PER_SERIES = 64_000
SERIES_NAMES = ["noisy_trend_1", "noisy_trend_2", "noisy_trend_3", "noisy_trend_4"]
EXPECTED_SOURCE_ROWS = ROWS_PER_SERIES * len(SERIES_NAMES)
GENERATION_CHUNK_SIZE = 64_000
COPY_TO_SQL_BATCH_ROWS = 128_000


def execute_ignore_missing(sql: str) -> None:
    try:
        cur.execute(sql)
    except Exception as exc:
        if "3807" not in str(exc):
            print(f"Ignoring cleanup error: {exc}")


def table_row_count(table_name: str) -> int | None:
    try:
        cur.execute(f"SELECT COUNT(*) FROM {table_name}")
        return int(cur.fetchone()[0])
    except Exception as exc:
        if "3807" in str(exc):
            return None
        raise


existing_source_rows = table_row_count(SOURCE_TABLE)
if existing_source_rows == EXPECTED_SOURCE_ROWS:
    print(f"Reusing {SOURCE_TABLE}: {existing_source_rows:,} rows already loaded")
else:
    if existing_source_rows is None:
        print(f"{SOURCE_TABLE} does not exist; generating {EXPECTED_SOURCE_ROWS:,} rows")
    else:
        print(
            f"{SOURCE_TABLE} has {existing_source_rows:,} rows; "
            f"rebuilding expected {EXPECTED_SOURCE_ROWS:,} rows"
        )
    execute_ignore_missing(f"DROP TABLE {SOURCE_TABLE}")
    cur.execute(f"""
CREATE MULTISET TABLE {SOURCE_TABLE}
(
    series_id VARCHAR(64) CHARACTER SET LATIN,
    seq_no    INTEGER,
    sample_ts TIMESTAMP(6),
    y         DOUBLE PRECISION
)
PRIMARY INDEX (series_id)
""")

    base_ts = pd.Timestamp("2026-01-01 00:00:00")
    rng = np.random.default_rng(20260630)
    t0 = time.perf_counter()
    inserted = 0
    upload_buffer = []
    buffered_rows = 0

    def flush_upload_buffer(force: bool = False) -> None:
        global inserted, buffered_rows, upload_buffer
        if not upload_buffer or (not force and buffered_rows < COPY_TO_SQL_BATCH_ROWS):
            return
        upload_df = pd.concat(upload_buffer, ignore_index=True)
        copy_to_sql(
            df=upload_df,
            table_name=SOURCE_TABLE,
            schema_name=DATA_DATABASE,
            if_exists="append",
            index=False,
            chunksize=COPY_TO_SQL_BATCH_ROWS,
            match_column_order=True,
        )
        inserted += len(upload_df)
        upload_buffer = []
        buffered_rows = 0
        print(f"Uploaded {inserted:,} rows")

    for series_idx, series_id in enumerate(SERIES_NAMES, start=1):
        slope = 0.000015 * series_idx
        amplitude = 0.7 + 0.15 * series_idx
        noise_scale = 0.12 + 0.03 * series_idx
        phase = series_idx * 0.7
        for start_seq in range(0, ROWS_PER_SERIES, GENERATION_CHUNK_SIZE):
            stop_seq = min(start_seq + GENERATION_CHUNK_SIZE, ROWS_PER_SERIES)
            seq = np.arange(start_seq, stop_seq, dtype=np.int32)
            y = np.empty(len(seq), dtype=float)

            rated_power = 3000 + 500 * series_idx

            state = (np.sin(seq / 20000.0) > 0).astype(float)
            baseline = state * rated_power
            noise = rng.normal(0, 50, len(seq))
            y[:] = baseline + noise

            n_bursts = 5

            for _ in range(n_bursts):
                center = rng.integers(0, len(seq))
                width = rng.integers(500, 5000)
                start = max(0, center - width // 2)
                end = min(len(seq), center + width // 2)
                local_seq = seq[start:end]
                freq = rng.uniform(2, 10)

                burst = (
                    rated_power * 0.8 * np.sin(local_seq / freq)
                    + rated_power * 0.5 * rng.normal(0, 1, len(local_seq))
                )

                if rng.random() < 0.5:
                    flip = ((local_seq % 2) * 2 - 1)
                    burst += flip * rated_power * 0.6

                y[start:end] = burst

            y = np.clip(y, 0, rated_power * 1.2)

            ts = base_ts + pd.to_timedelta(seq, unit="s")
            chunk_df = pd.DataFrame(
                {
                    "series_id": series_id,
                    "seq_no": seq,
                    "sample_ts": ts,
                    "y": y.astype(float),
                }
            )
            upload_buffer.append(chunk_df)
            buffered_rows += len(chunk_df)
            flush_upload_buffer()

    flush_upload_buffer(force=True)
    con.commit()
    print(f"Rows uploaded: {inserted:,} in {time.perf_counter() - t0:.1f}s")

cur.execute(f"SELECT COUNT(*) FROM {SOURCE_TABLE}")
print(f"Source rows available: {cur.fetchone()[0]:,}")


## 4. Try `TD_PLOT` on the Raw 4M Rows

This mirrors the raw SQL plot attempt in the reference notebook. Depending on
the system, this may fail because of UAF image/point limits, or it may
unexpectedly succeed.

In [ ]:
plot_compressor = TeradataTimeSeriesCompressor(
    CompressorSpec(
        id_columns=["series_id"],
        time_column="sample_ts",
        value_column="y",
        time_type="timestamp",
        time_unit="second",
    ),
    operator_database=OPERATOR_DATABASE,
)

raw_plot_sql = plot_compressor.td_plot_sql(
    schema_name=DATA_DATABASE,
    table_name=SOURCE_TABLE,
    title="Raw 4M noisy trend rows",
    width=2048,
    height=1536,
)
print(raw_plot_sql)

raw_plot_error = None
t0 = time.perf_counter()
try:
    cur.execute(raw_plot_sql)
    raw_plot_rows = cur.fetchall()
    print(f"Unexpected success: TD_PLOT returned {len(raw_plot_rows)} row(s).")
    raw_png = save_td_plot_png(raw_plot_rows, ASSET_DIR / "03_td_plot_raw_4m.png")
    display(Image(filename=str(raw_png)))
except Exception as exc:
    raw_plot_error = str(exc)
    print(f"Expected TD_PLOT failure after {time.perf_counter() - t0:.1f}s:")
    print(raw_plot_error.splitlines()[0])


## 5. Compress for Visualization

The compressed table is created explicitly, then populated with `INSERT INTO`
from the package-generated table-operator SQL. The source still has four
TD_PLOT series, but compression is partitioned into 100K-row windows to keep
each table-operator partition inside the current C operator's buffering range.
The compressor now uses `sample_ts` as the time axis, then joins the selected timestamp points back to the source table to retain `seq_no` for local overlays.


In [ ]:
compression_compressor = TeradataTimeSeriesCompressor(
    CompressorSpec(
        id_columns=["series_id"],
        time_column="sample_ts",
        value_column="y",
        time_type="timestamp",
        time_unit="second",
    ),
    operator_database=OPERATOR_DATABASE,
)

distance_for_pixel_box = float(np.sqrt(2.0))
selected_points_sql = compression_compressor.compress(
    data_database=DATA_DATABASE,
    table_name=SOURCE_TABLE,
    columns=[
        "series_id",
        "sample_ts",
        "y",
    ],
    parameters=visual_fidelity(width_px=320, height_px=200, distance_max=distance_for_pixel_box),
    as_dataframe=False,
)

operator_sql = f"""
SELECT
    src.series_id,
    src.seq_no,
    src.sample_ts,
    src.y,
    selected.compression_importance,
    selected.compression_distance
FROM (
{selected_points_sql}
) AS selected
JOIN {DATA_DATABASE}.{SOURCE_TABLE} AS src
  ON src.series_id = selected.series_id
 AND src.sample_ts = selected.sample_ts
"""

execute_ignore_missing(f"DROP TABLE {COMPRESSED_TABLE}")
cur.execute(f"""
CREATE MULTISET TABLE {COMPRESSED_TABLE}
(
    series_id VARCHAR(64) CHARACTER SET LATIN,
    seq_no INTEGER,
    sample_ts TIMESTAMP(6),
    y DOUBLE PRECISION,
    compression_importance INTEGER,
    compression_distance FLOAT
)
PRIMARY INDEX (series_id, sample_ts)
""")

print(operator_sql)
t0 = time.perf_counter()
cur.execute(f"INSERT INTO {COMPRESSED_TABLE} {operator_sql}")
con.commit()
compression_seconds = time.perf_counter() - t0

cur.execute(f"SELECT COUNT(*) FROM {SOURCE_TABLE}")
original_count = cur.fetchone()[0]
cur.execute(f"SELECT COUNT(*) FROM {COMPRESSED_TABLE}")
compressed_count = cur.fetchone()[0]
print(f"Compressed {original_count:,} rows to {compressed_count:,} rows in {compression_seconds:.1f}s")
print(f"Compression ratio: {original_count / compressed_count:.2f}x")
cur.execute(f"""
SELECT series_id, COUNT(*) AS compressed_rows
FROM {COMPRESSED_TABLE}
GROUP BY 1
ORDER BY 1
""")
display(pd.DataFrame(cur.fetchall(), columns=["series_id", "compressed_rows"]))


## 6. `TD_PLOT` on the Compressed Table

`TD_PLOT` receives the smaller compressed table, but the retained points were
chosen geometrically for visual fidelity.

In [ ]:
compressed_plot_sql = plot_compressor.td_plot_sql(
    schema_name=DATA_DATABASE,
    table_name=COMPRESSED_TABLE,
    title="Compressed noisy trends by ts_compress_to",
    width=1024,
    height=768,
)
print(compressed_plot_sql)

t0 = time.perf_counter()
cur.execute(compressed_plot_sql)
plot_rows = cur.fetchall()
plot_seconds = time.perf_counter() - t0
print(f"Compressed TD_PLOT returned {len(plot_rows)} row(s) in {plot_seconds:.1f}s")

compressed_png = save_td_plot_png(plot_rows, ASSET_DIR / "03_td_plot_compressed_4m.png")
display(Image(filename=str(compressed_png)))


## 7. Local Overlay Check

This local plot is only a sanity check. It overlays the first source series
against the compressed points retained for that same series.

In [ ]:
overlay_series = SERIES_NAMES[0]
cur.execute(f"""
SELECT series_id, seq_no, sample_ts, y
FROM {SOURCE_TABLE}
WHERE series_id = '{overlay_series}'
  AND seq_no BETWEEN 0 AND 25000
ORDER BY seq_no
""")
original_df = pd.DataFrame(cur.fetchall(), columns=["series_id", "seq_no", "sample_ts", "y"])

cur.execute(f"""
SELECT series_id, seq_no, sample_ts, y
FROM {COMPRESSED_TABLE}
WHERE series_id = '{overlay_series}'
  AND seq_no BETWEEN 0 AND 25000
ORDER BY seq_no
""")
compressed_df = pd.DataFrame(cur.fetchall(), columns=["series_id", "seq_no", "sample_ts", "y"])

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(original_df["sample_ts"], original_df["y"], color="#5b8db8", alpha=0.35, linewidth=0.7, label="original")
ax.plot(compressed_df["sample_ts"], compressed_df["y"], color="#0f766e", linewidth=1.3, marker="o", markersize=2.5, label="compressed")
ax.set_title(f"Original vs compressed data used by TD_PLOT: {overlay_series}")
ax.set_ylabel("y")
ax.grid(alpha=0.25)
ax.legend(loc="best")
fig.tight_layout()
overlay_png = ASSET_DIR / "03_original_vs_compressed_td_plot.png"
fig.savefig(overlay_png, dpi=150)
plt.close(fig)
display(Image(filename=str(overlay_png)))


## 8. Close Connection

In [ ]:
remove_context()
con.close()
print("Connection closed.")
